In [29]:
import polars as pl
from pathlib import Path
import json
from pprint import pprint

# -----------------------------
# CONFIG
# -----------------------------
all_data = "/home/fabi/cern_db/dihiggs_consolidated/dihiggs_lake_phys_only.parquet"
subset = "/home/fabi/wt_dihiggs_exploratory/mlpython/2603/temp_subspace_BR1e-1_DW1e-11.parquet"
PARQUET_PATH = Path(subset)

# Este parquet ya debería ser phys-only.
# Ponlo en True solo si alguna vez apuntas al raw parquet.
APPLY_PHYS_FILTER = False

# Flags teóricos esperados
FLAG_COLS = ["positivity_ok", "unitarity_ok", "perturbativity_ok"]


In [30]:

# Tamaños seguros para presentación
DEFAULT_HEAD = 20
DEFAULT_LIMIT = 50
DEFAULT_TOPK = 20

# Streaming-safe collect
def scollect(lf: pl.LazyFrame) -> pl.DataFrame:
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)
    except Exception:
        return lf.collect()

def resolve_col(schema_names, aliases):
    lower_map = {c.lower(): c for c in schema_names}
    for a in aliases:
        if a.lower() in lower_map:
            return lower_map[a.lower()]
    for c in schema_names:
        cl = c.lower()
        if any(tok.lower() in cl for tok in aliases):
            return c
    return None

def flag_true_expr(col_name: str) -> pl.Expr:
    as_num = pl.col(col_name).cast(pl.Float64, strict=False)
    as_txt = (
        pl.col(col_name)
        .cast(pl.Utf8, strict=False)
        .str.strip_chars()
        .str.to_lowercase()
    )
    return (as_num >= 0.5) | as_txt.is_in(["1", "1.0", "true", "t"])

print(f"Parquet exists: {PARQUET_PATH.exists()}")
print(f"Parquet path  : {PARQUET_PATH}")

Parquet exists: True
Parquet path  : /home/fabi/wt_dihiggs_exploratory/mlpython/2603/temp_subspace_BR1e-1_DW1e-11.parquet


In [31]:
lf_base = pl.scan_parquet(PARQUET_PATH)
schema = lf_base.collect_schema()
schema_names = schema.names()

print(f"Número de columnas: {len(schema_names)}")
print("Columnas:")
for c in schema_names:
    print(" -", c)

Número de columnas: 29
Columnas:
 - m_phi
 - mA
 - alpha
 - beta
 - lambda6
 - lambda7
 - m12
 - sin_ba
 - tan_beta
 - positivity_ok
 - unitarity_ok
 - perturbativity_ok
 - width_bb
 - width_tautau
 - width_WW
 - width_ZZ
 - width_gaga
 - width_Zga
 - width_gg
 - width_hh
 - total_width
 - br_gaga
 - lam1
 - computed_lam1
 - lam2
 - computed_lam2
 - lam3
 - lam4
 - lam5


In [32]:
# Resolver columnas clave
COL_MPHI = resolve_col(schema_names, ["m_phi", "mH", "mh2"])
COL_MA = resolve_col(schema_names, ["mA"])
COL_L6 = resolve_col(schema_names, ["lambda6", "lambda_6", "lam6"])
COL_TB = resolve_col(schema_names, ["tan_beta", "tanbeta"])
COL_BR = resolve_col(schema_names, ["br_gaga", "branching_ratio", "br"])
COL_WTOT = resolve_col(schema_names, ["total_width", "total_decay_width", "w_total"])

print("Columnas resueltas:")
print("  m_phi      =", COL_MPHI)
print("  mA         =", COL_MA)
print("  lambda6    =", COL_L6)
print("  tan_beta   =", COL_TB)
print("  br_gaga    =", COL_BR)
print("  total_width=", COL_WTOT)

needed_cols = [c for c in [COL_MPHI, COL_MA, COL_L6, COL_TB, COL_BR, COL_WTOT] if c is not None]
present_flags = [c for c in FLAG_COLS if c in schema_names]

if APPLY_PHYS_FILTER:
    needed_cols += [c for c in present_flags if c not in needed_cols]

lf = lf_base.select(needed_cols)

# filtro lambda_6 > 0

if APPLY_PHYS_FILTER and present_flags:
    expr = None
    for c in present_flags:
        e = flag_true_expr(c)
        expr = e if expr is None else (expr & e)
    lf = lf.filter(expr)

print("Columnas de trabajo:")
print(needed_cols)

Columnas resueltas:
  m_phi      = m_phi
  mA         = mA
  lambda6    = lambda6
  tan_beta   = tan_beta
  br_gaga    = br_gaga
  total_width= total_width
Columnas de trabajo:
['m_phi', 'mA', 'lambda6', 'tan_beta', 'br_gaga', 'total_width']


In [33]:
# Resolver columnas clave
COL_MPHI = resolve_col(schema_names, ["m_phi", "mH", "mh2"])
COL_MA = resolve_col(schema_names, ["mA"])
COL_L6 = resolve_col(schema_names, ["lambda6", "lambda_6", "lam6"])
COL_TB = resolve_col(schema_names, ["tan_beta", "tanbeta"])
COL_BR = resolve_col(schema_names, ["br_gaga", "branching_ratio", "br"])
COL_WTOT = resolve_col(schema_names, ["total_width", "total_decay_width", "w_total"])

print("Columnas resueltas:")
print("  m_phi      =", COL_MPHI)
print("  mA         =", COL_MA)
print("  lambda6    =", COL_L6)
print("  tan_beta   =", COL_TB)
print("  br_gaga    =", COL_BR)
print("  total_width=", COL_WTOT)

needed_cols = [c for c in [COL_MPHI, COL_MA, COL_L6, COL_TB, COL_BR, COL_WTOT] if c is not None]
present_flags = [c for c in FLAG_COLS if c in schema_names]

if APPLY_PHYS_FILTER:
    needed_cols += [c for c in present_flags if c not in needed_cols]

lf = lf_base.select(needed_cols)

if APPLY_PHYS_FILTER and present_flags:
    expr = None
    for c in present_flags:
        e = flag_true_expr(c)
        expr = e if expr is None else (expr & e)
    lf = lf.filter(expr)

print("Columnas de trabajo:")
print(needed_cols)

Columnas resueltas:
  m_phi      = m_phi
  mA         = mA
  lambda6    = lambda6
  tan_beta   = tan_beta
  br_gaga    = br_gaga
  total_width= total_width
Columnas de trabajo:
['m_phi', 'mA', 'lambda6', 'tan_beta', 'br_gaga', 'total_width']


In [34]:
def make_filter(filters: dict | None = None) -> pl.Expr | None:
    """
    filters ejemplo:
    {"mA": 300.0, "lambda6": 0.001, "tan_beta": 1e6}
    """
    if not filters:
        return None

    expr = None
    for k, v in filters.items():
        e = (pl.col(k) == v)
        expr = e if expr is None else (expr & e)
    return expr


def q_count(filters: dict | None = None):
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)
    return scollect(q.select(pl.len().alias("rows")))


def q_preview(cols=None, filters: dict | None = None, n: int = DEFAULT_HEAD):
    cols = cols or needed_cols
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)
    return scollect(q.select(cols).limit(n))


def q_stats(col: str, filters: dict | None = None):
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)

    return scollect(
        q.select(
            pl.len().alias("rows"),
            pl.col(col).null_count().alias("nulls"),
            pl.col(col).min().alias("min"),
            pl.col(col).max().alias("max"),
            pl.col(col).mean().alias("mean"),
            pl.col(col).std().alias("std"),
        )
    )


def q_unique_small(col: str, filters: dict | None = None, limit: int = 100):
    """
    Seguro para columnas de cardinalidad pequeña o moderada.
    No usar en columnas continuas gigantes.
    """
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)

    return scollect(
        q.select(col)
         .drop_nulls()
         .unique()
         .sort(col)
         .limit(limit)
    )


def q_top_groups(group_cols: list[str], filters: dict | None = None, topk: int = DEFAULT_TOPK):
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)

    return scollect(
        q.group_by(group_cols)
         .len()
         .sort("len", descending=True)
         .limit(topk)
    )


def q_ctau_preview(filters: dict | None = None, n: int = DEFAULT_HEAD):
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)

    q = q.with_columns(
        pl.when(pl.col(COL_WTOT).is_not_null() & (~pl.col(COL_WTOT).is_nan()) & (pl.col(COL_WTOT) > 0))
        .then((6.582119569e-25 * 2.99792458e11) / pl.col(COL_WTOT).cast(pl.Float64))
        .otherwise(None)
        .alias("c_tau_mm")
    )

    return scollect(
        q.select([COL_MPHI, COL_MA, COL_L6, COL_TB, COL_BR, COL_WTOT, "c_tau_mm"])
         .limit(n)
    )

In [35]:
print("Filas totales:")
display(q_count())

print("Valores únicos pequeños de mA:")
display(q_unique_small(COL_MA, limit=50))

print("Valores únicos pequeños de lambda6:")
display(q_unique_small(COL_L6, limit=50))

print("Valores únicos pequeños de tan_beta:")
display(q_unique_small(COL_TB, limit=50))

Filas totales:


rows
u32
15064


Valores únicos pequeños de mA:


mA
f64
300.0


Valores únicos pequeños de lambda6:


lambda6
f32
0.001


Valores únicos pequeños de tan_beta:


tan_beta
f32
500000.0
1e6


In [36]:
display(q_top_groups([COL_MA, COL_L6], topk=20))

mA,lambda6,len
f64,f32,u32
300.0,0.001,15064


In [37]:
display(q_top_groups([COL_MA, COL_L6, COL_TB], topk=20))

mA,lambda6,tan_beta,len
f64,f32,f32,u32
300.0,0.001,500000.0,11985
300.0,0.001,1e6,3079


In [38]:
my_filters = {
    COL_MA: 300.0,
    COL_L6: 0.0010000000474974513,
}

print("Número de filas en el slice:")
display(q_count(my_filters))

print("tan_beta presentes en ese slice:")
display(q_unique_small(COL_TB, filters=my_filters, limit=50))

print("Preview del slice:")
display(q_preview(filters=my_filters, n=20))

Número de filas en el slice:


rows
u32
15064


tan_beta presentes en ese slice:


tan_beta
f32
500000.0
1e6


Preview del slice:


m_phi,mA,lambda6,tan_beta,br_gaga,total_width
f32,f64,f32,f32,f32,f32
130.0,300.0,0.001,1e6,0.444924,7.0291e-15
130.0,300.0,0.001,1e6,0.444925,7.0292e-15
130.0,300.0,0.001,1e6,0.444926,7.0292e-15
130.0,300.0,0.001,1e6,0.444928,7.0292e-15
130.0,300.0,0.001,1e6,0.444929,7.0292e-15
…,…,…,…,…,…
130.0,300.0,0.001,1e6,0.444941,7.0294e-15
130.0,300.0,0.001,1e6,0.444942,7.0294e-15
130.0,300.0,0.001,1e6,0.444943,7.0294e-15


In [39]:
display(q_stats(COL_BR, filters=my_filters))
display(q_stats(COL_WTOT, filters=my_filters))
display(q_stats(COL_MPHI, filters=my_filters))

rows,nulls,min,max,mean,std
u32,u32,f32,f32,f32,f32
15064,0,0.173562,0.496808,0.282592,0.097734


rows,nulls,min,max,mean,std
u32,u32,f32,f32,f32,f32
15064,0,7.0291e-15,5.2497e-14,2.3931e-14,1.1058e-14


rows,nulls,min,max,mean,std
u32,u32,f32,f32,f32,f32
15064,0,130.0,221.428604,159.410522,24.807453


In [40]:
display(q_ctau_preview(filters=my_filters, n=20))

m_phi,mA,lambda6,tan_beta,br_gaga,total_width,c_tau_mm
f32,f64,f32,f32,f32,f32,f64
130.0,300.0,0.001,1e6,0.444924,7.0291e-15,28.07269
130.0,300.0,0.001,1e6,0.444925,7.0292e-15,28.072622
130.0,300.0,0.001,1e6,0.444926,7.0292e-15,28.072555
130.0,300.0,0.001,1e6,0.444928,7.0292e-15,28.072487
130.0,300.0,0.001,1e6,0.444929,7.0292e-15,28.072414
…,…,…,…,…,…,…
130.0,300.0,0.001,1e6,0.444941,7.0294e-15,28.071663
130.0,300.0,0.001,1e6,0.444942,7.0294e-15,28.071599
130.0,300.0,0.001,1e6,0.444943,7.0294e-15,28.071528


In [41]:
def q_slice_summary(filters: dict | None = None):
    expr = make_filter(filters)
    q = lf
    if expr is not None:
        q = q.filter(expr)

    out = scollect(
        q.select(
            pl.len().alias("rows"),
            pl.col(COL_MPHI).min().alias("mphi_min"),
            pl.col(COL_MPHI).max().alias("mphi_max"),
            pl.col(COL_MA).min().alias("mA_min"),
            pl.col(COL_MA).max().alias("mA_max"),
            pl.col(COL_L6).min().alias("lambda6_min"),
            pl.col(COL_L6).max().alias("lambda6_max"),
            pl.col(COL_TB).min().alias("tanbeta_min"),
            pl.col(COL_TB).max().alias("tanbeta_max"),
            pl.col(COL_BR).min().alias("br_min"),
            pl.col(COL_BR).max().alias("br_max"),
            pl.col(COL_WTOT).min().alias("width_min"),
            pl.col(COL_WTOT).max().alias("width_max"),
        )
    )
    return out

display(q_slice_summary())
display(q_slice_summary(my_filters))

rows,mphi_min,mphi_max,mA_min,mA_max,lambda6_min,lambda6_max,tanbeta_min,tanbeta_max,br_min,br_max,width_min,width_max
u32,f32,f32,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32
15064,130.0,221.428604,300.0,300.0,0.001,0.001,500000.0,1e6,0.173562,0.496808,7.0291e-15,5.2497e-14


rows,mphi_min,mphi_max,mA_min,mA_max,lambda6_min,lambda6_max,tanbeta_min,tanbeta_max,br_min,br_max,width_min,width_max
u32,f32,f32,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32
15064,130.0,221.428604,300.0,300.0,0.001,0.001,500000.0,1e6,0.173562,0.496808,7.0291e-15,5.2497e-14


In [42]:
# Cambia aquí según la pregunta del comité
ask_filters = {
    COL_MA: 300.0,
    COL_L6: 0.0010000000474974513,
    # COL_TB: 1_000_000.0,
}

display(q_count(ask_filters))
display(q_unique_small(COL_TB, filters=ask_filters, limit=20))
display(q_stats(COL_BR, filters=ask_filters))
display(q_ctau_preview(filters=ask_filters, n=10))

rows
u32
15064


tan_beta
f32
500000.0
1e6


rows,nulls,min,max,mean,std
u32,u32,f32,f32,f32,f32
15064,0,0.173562,0.496808,0.282592,0.097734


m_phi,mA,lambda6,tan_beta,br_gaga,total_width,c_tau_mm
f32,f64,f32,f32,f32,f32,f64
130.0,300.0,0.001,1e6,0.444924,7.0291e-15,28.07269
130.0,300.0,0.001,1e6,0.444925,7.0292e-15,28.072622
130.0,300.0,0.001,1e6,0.444926,7.0292e-15,28.072555
130.0,300.0,0.001,1e6,0.444928,7.0292e-15,28.072487
130.0,300.0,0.001,1e6,0.444929,7.0292e-15,28.072414
130.0,300.0,0.001,1e6,0.44493,7.0292e-15,28.072347
130.0,300.0,0.001,1e6,0.444931,7.0292e-15,28.072279
130.0,300.0,0.001,1e6,0.444932,7.0293e-15,28.072211
130.0,300.0,0.001,1e6,0.444933,7.0293e-15,28.072144
